# Part 19: Archaeology — Firehose Consumer

The ATProto firehose is a real-time event stream that carries every record write, delete, and
identity update across the network. Consuming it requires understanding the event types, sequencing,
and backpressure handling.

This notebook models a firehose consumer that processes events from a simulated stream, tracking the
cursor for replay.

**What you'll learn:**

- Firehose event types (repo commits, handles, identities, tombstones)
- Sequencing and cursor management
- Backpressure via buffer management
- Tombstone (delete) handling

**Prerequisites:** Parts 12, 16, 18

**Estimated Time:** 20-25 minutes

## Step 1: Firehose Event Types

The firehose emits events as DAG-CBOR frames over WebSocket. Each event has a type discriminator and
a sequence number for ordering and replay.

| Type         | Meaning                                    |
| ------------ | ------------------------------------------ |
| `#commit`    | Repo write (create, update, delete record) |
| `#handle`    | Handle change                              |
| `#identity`  | DID document change                        |
| `#tombstone` | Repo deleted                               |

In [ ]:
// Firehose event model
@interface FirehoseEvent : NSObject
@property (nonatomic, assign) int seq;
@property (nonatomic, strong) NSString *eventType;
@property (nonatomic, strong) NSString *did;
- (NSString *)description;
@end

@implementation FirehoseEvent
- (NSString *)description {
    return [NSString stringWithFormat:@"#%d %@ %@", _seq, _eventType, _did];
}
@end

// Create sample events
FirehoseEvent *e1 = [[FirehoseEvent alloc] init];
e1.seq = 1;
e1.eventType = @"#commit";
e1.did = @"did:plc:abc123";
NSLog(@"Event: %@", [e1 description]);

## Step 2: Cursor and Replay

The firehose consumer tracks a cursor (sequence number) so it can resume after disconnection. On
reconnect, it requests all events with seq > cursor.

In [ ]:
// Firehose consumer with cursor tracking
@interface FirehoseConsumer : NSObject
@property (nonatomic, assign) int cursor;
@property (nonatomic, strong) NSMutableArray *buffer;
- (void)processEvent:(FirehoseEvent *)event;
- (int)eventsProcessed;
@end

@implementation FirehoseConsumer
- (instancetype)init {
    self = [super init];
    if (self) {
        _cursor = 0;
        _buffer = [NSMutableArray array];
    }
    return self;
}
- (void)processEvent:(FirehoseEvent *)event {
    if ([event seq] <= _cursor) {
        NSLog(@"Skipping old event #%d", [event seq]);
        return;
    }
    [_buffer addObject:event];
    _cursor = [event seq];
    NSLog(@"Processed: %@", [event description]);
}
- (int)eventsProcessed {
    return [_buffer count];
}
@end

FirehoseConsumer *consumer = [[FirehoseConsumer alloc] init];
FirehoseEvent *e2 = [[FirehoseEvent alloc] init];
e2.seq = 5;
e2.eventType = @"#commit";
e2.did = @"did:plc:xyz789";
[consumer processEvent:e2];

// Duplicate should be skipped
[consumer processEvent:e2];
NSLog(@"Total processed: %d", [consumer eventsProcessed]);

## Step 3: Tombstone Handling

When a repository is deleted, a tombstone event is emitted. The consumer must mark all records from
that DID as deleted.

In [ ]:
// Tombstone-aware consumer
@interface TombstoneConsumer : NSObject
@property (nonatomic, strong) NSMutableArray *deletedDIDs;
@property (nonatomic, assign) int cursor;
- (void)processEvent:(FirehoseEvent *)event;
- (BOOL)isDIDDeleted:(NSString *)did;
@end

@implementation TombstoneConsumer
- (instancetype)init {
    self = [super init];
    if (self) {
        _deletedDIDs = [NSMutableArray array];
        _cursor = 0;
    }
    return self;
}
- (void)processEvent:(FirehoseEvent *)event {
    _cursor = [event seq];
    if ([[event eventType] isEqualToString:@"#tombstone"]) {
        [_deletedDIDs addObject:[event did]];
        NSLog(@"Tombstone: %@ deleted", [event did]);
    } else if ([self isDIDDeleted:[event did]]) {
        NSLog(@"Skipping event for deleted DID: %@", [event did]);
    } else {
        NSLog(@"Processed: %@", [event description]);
    }
}
- (BOOL)isDIDDeleted:(NSString *)did {
    return [_deletedDIDs containsObject:did];
}
@end

TombstoneConsumer *tc = [[TombstoneConsumer alloc] init];

// Normal event
FirehoseEvent *commit = [[FirehoseEvent alloc] init];
commit.seq = 10;
commit.eventType = @"#commit";
commit.did = @"did:plc:deleted1";
[tc processEvent:commit];

// Tombstone
FirehoseEvent *tomb = [[FirehoseEvent alloc] init];
tomb.seq = 11;
tomb.eventType = @"#tombstone";
tomb.did = @"did:plc:deleted1";
[tc processEvent:tomb];

// Post-tombstone event should be skipped
FirehoseEvent *post = [[FirehoseEvent alloc] init];
post.seq = 12;
post.eventType = @"#commit";
post.did = @"did:plc:deleted1";
[tc processEvent:post];

## Step 4: Production Anchor

In Garazyk, the firehose consumer is in `Sources/Firehose/`:

- `FirehoseConsumer.h` — WebSocket connection and event dispatch
- `FirehoseCursor.h` — persistent cursor storage
- `RepoCommit.h` — DAG-CBOR commit deserialization

The production consumer handles reconnection, backpressure, and parallel event processing via
operation queues.

## Exercise

Add a `drainBuffer` method to `FirehoseConsumer` that processes all buffered events and returns the
count. After draining, the buffer should be empty.

In [ ]:
// Exercise: implement drainBuffer
// Your code here...

## Solution

In [ ]:
// Solution: drainBuffer processes and clears the buffer
@interface BufferConsumer : NSObject
@property (nonatomic, strong) NSMutableArray *buffer;
- (void)addEvent:(NSString *)event;
- (int)drainBuffer;
@end

@implementation BufferConsumer
- (instancetype)init {
    self = [super init];
    if (self) { _buffer = [NSMutableArray array]; }
    return self;
}
- (void)addEvent:(NSString *)event {
    [_buffer addObject:event];
}
- (int)drainBuffer {
    int count = [_buffer count];
    [_buffer removeAllObjects];
    return count;
}
@end

BufferConsumer *bc = [[BufferConsumer alloc] init];
[bc addEvent:@"commit"];
[bc addEvent:@"handle"];
[bc addEvent:@"identity"];
int drained = [bc drainBuffer];
NSLog(@"Drained %d events", drained);

## What to Read Next

You now understand firehose consumption. Next:

- **Part 20: Archaeology — Repo Sync** — full repository synchronization
- **Part 16: Firehose and Sync** — the production firehose implementation